# EVAL.ipynb — Phi-2 MLP vs. Phi-2 Selector (Colab)

Evaluates two trained Prismatic VLM variants on held-out captioning sets:

1. **PHI2-MLP** — `arch_specifier="...gelu-mlp"`, passes all visual tokens.
2. **PHI2-Selector** — `arch_specifier="...selector"`, learned top-K selection.

Pipeline: setup → mount Drive → paths/rsync/install → login → untar LAION → untar COCO-500 → split LAION held-out → selector diagnostics (heatmaps + oracle agreement) → end-to-end captioning (CIDEr + CLIPScore) → qualitative examples → save to Drive.

Same Drive layout as `ABLATIONS.ipynb`: source rsync'd from `/content/drive/MyDrive/targe-prismatic-vlms` to `/content/repo`, datasets staged on the VM SSD under `/content/data`, durable outputs written back to Drive under `runs/<RUN_ID>/`.

## 1. Setup: GPU check + mount Drive

In [ ]:
print("== GPU ==")
!nvidia-smi -L || echo "(no GPU detected)"
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader || true

print("\n== Drive ==")
from google.colab import drive
drive.mount('/content/drive')
print("Drive mounted at /content/drive")

## 2. Filepaths + rsync + install

Single source of truth for every path used here.

- **Drive** = `/content/drive/MyDrive/...` — persistent, slow (FUSE). Holds the authoritative source code and durable run outputs (checkpoints, results JSON).
- **Local** = `/content/...` — VM SSD, lost on session teardown. Holds the source mirror, datasets, and HF cache.

This cell mirrors the repo from Drive → local SSD with rsync (same as ABLATIONS) and runs `pip install -e .` from the mirror.

In [ ]:
import os, time
from pathlib import Path

# ─── H100 / Ampere+ perf knobs ──────────────────────────────────────────
try:
    import torch
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32       = True
    torch.backends.cudnn.benchmark        = True
    torch.set_float32_matmul_precision("high")
    if torch.cuda.is_available():
        print(f"[gpu] {torch.cuda.get_device_name(0)}  (TF32 on, cudnn.benchmark on)")
except Exception as _e:
    print(f"[gpu] perf knobs skipped: {_e}")

# ─── Tunable knobs ──────────────────────────────────────────────────────
RUN_ID_MLP      = "phi2-mlp-75"
RUN_ID_SELECTOR = "phi2-selector-75"
EVAL_TAG        = "phi2-eval-75"   # subdir for this notebook's outputs on Drive

N_HELDOUT  = 500            # LAION held-out size
N_COCO     = 500
SPLIT_SEED = 7

K_VALUES        = [16, 32, 64, 128]    # K sweep for selector evals
MAX_NEW_TOKENS  = 32
VIZ_N_SAMPLES   = 6
DEMO_K_FOR_GEN  = 64

# ─── Drive (persistent) ─────────────────────────────────────────────────
DRIVE_ROOT        = Path("/content/drive/MyDrive/targe-prismatic-vlms")
CKPT_DIR_MLP      = DRIVE_ROOT / "runs" / RUN_ID_MLP
CKPT_DIR_SELECTOR = DRIVE_ROOT / "runs" / RUN_ID_SELECTOR
EVAL_OUT_DIR      = DRIVE_ROOT / "runs" / EVAL_TAG
RESULTS_JSON      = EVAL_OUT_DIR / "eval_results.json"
EVAL_LOG          = EVAL_OUT_DIR / "eval.log"

# ─── Local (VM SSD scratch) ─────────────────────────────────────────────
LOCAL_ROOT   = Path("/content")
REPO_DIR     = LOCAL_ROOT / "repo"
TAR_PATH     = LOCAL_ROOT / "train_subset_75.tar"          # LAION shard (same as ABLATIONS)
DATA_DIR     = LOCAL_ROOT / "data" / "download" / "llava-laion-cc-sbu-558k"
CHAT_JSON    = DATA_DIR / "chat.json"                      # training subset (after split)
CHAT_FULL    = DATA_DIR / "chat_full.json"                 # original pre-split sentinel
HELDOUT_JSON = DATA_DIR / "chat_heldout.json"              # 500 held-out
COCO_TAR     = LOCAL_ROOT / "coco_500.tar.gz"              # upload manually
COCO_DIR     = LOCAL_ROOT / "data" / "coco_500"            # extracted COCO-500
HF_CACHE     = LOCAL_ROOT / "hf_cache"

os.chdir(LOCAL_ROOT)
for d in (DATA_DIR.parent, COCO_DIR.parent, HF_CACHE, EVAL_OUT_DIR):
    d.mkdir(parents=True, exist_ok=True)
os.environ["HF_HOME"]      = str(HF_CACHE)
os.environ["HF_HUB_CACHE"] = str(HF_CACHE / "hub")

# ─── Mirror source Drive → local SSD ────────────────────────────────────
t0 = time.time()
REPO_DIR.mkdir(parents=True, exist_ok=True)
!rsync -a \
    --exclude='runs' \
    --exclude='data' \
    --exclude='hf_cache' \
    --exclude='*.tar' \
    --exclude='*.tar.gz' \
    --exclude='__pycache__' \
    --exclude='*.pyc' \
    --exclude='.git' \
    --exclude='*.egg-info' \
    --exclude='build' \
    --exclude='dist' \
    --exclude='.ipynb_checkpoints' \
    "{DRIVE_ROOT}/" "{REPO_DIR}/"
print(f"[mirror] {time.time()-t0:.1f}s")

PYPROJECT = REPO_DIR / "pyproject.toml"
assert PYPROJECT.exists(), (
    f"rsync finished but {PYPROJECT} is missing — likely a Drive FUSE crash mid-copy.\n"
    "Fix: re-mount Drive ( drive.mount('/content/drive', force_remount=True) ),\n"
    "     then `!rm -rf /content/repo` and re-run this cell."
)

%cd {REPO_DIR}
INSTALLED_MARKER = REPO_DIR / ".pip_installed_eval"
if not INSTALLED_MARKER.exists():
    t0 = time.time()
    !pip install -e . --no-build-isolation --quiet
    !pip install -q pycocoevalcap open-clip-torch
    INSTALLED_MARKER.touch()
    print(f"[install] {time.time()-t0:.1f}s")
else:
    print("[install] cached  (remove /content/repo/.pip_installed_eval to force reinstall)")

# pycocoevalcap shells out to a JAR; Colab ships Java by default but verify.
import shutil
if shutil.which("java") is None:
    print("[warn] `java` not on PATH; installing default-jre-headless (CIDEr needs it)")
    !apt-get install -y -qq default-jre-headless
else:
    print(f"[java] OK -> {shutil.which('java')}")

print("\n" + "─" * 60)
print(f"Drive source     : {DRIVE_ROOT}")
print(f"MLP ckpt dir     : {CKPT_DIR_MLP}      ({'exists' if CKPT_DIR_MLP.exists() else 'MISSING'})")
print(f"Selector ckpt dir: {CKPT_DIR_SELECTOR} ({'exists' if CKPT_DIR_SELECTOR.exists() else 'MISSING'})")
print(f"Eval output dir  : {EVAL_OUT_DIR}")
print(f"Local repo       : {REPO_DIR}")
print(f"Local LAION tar  : {TAR_PATH}    ({'exists' if TAR_PATH.exists() else 'MISSING'})")
print(f"Local COCO tar   : {COCO_TAR}    ({'exists' if COCO_TAR.exists() else 'UPLOAD ME'})")
print(f"Local data dir   : {DATA_DIR}    ({'extracted' if CHAT_JSON.exists() or CHAT_FULL.exists() else 'not yet'})")
print(f"Local COCO dir   : {COCO_DIR}    ({'extracted' if (COCO_DIR / 'index_500.json').exists() else 'not yet'})")
print(f"HF cache         : {HF_CACHE}")

# Soft check — don't block here if checkpoints haven't been trained yet.
for label, p in [("MLP", CKPT_DIR_MLP), ("Selector", CKPT_DIR_SELECTOR)]:
    ck = p / "checkpoints" / "latest-checkpoint.pt"
    if not ck.exists():
        print(f"[warn] {label} checkpoint not found at {ck}")

## 3. Login (Hugging Face)

Phi-2 + CLIP-ViT downloads cleanly through public HF Hub, but logging in keeps rate-limited paths happy and matches the rest of the project.

In [ ]:
import os
try:
    from google.colab import userdata
    hf_token = userdata.get('hf_token')
except Exception:
    hf_token = os.environ.get("HF_TOKEN")

if hf_token:
    from huggingface_hub import login, whoami
    login(token=hf_token)
    os.environ["HF_TOKEN"] = hf_token
    who = whoami()
    print(f"[hf] logged in as: {who.get('name')}")
else:
    print("[hf] no token — proceeding anonymously (fine for public models)")

## 4. Untar LAION-558K shard

Idempotent: skips if `chat.json` (or its split sentinel `chat_full.json`) is already present. Extracts to `/content/data/download/llava-laion-cc-sbu-558k/`.

If you trained on Colab, `train_subset_75.tar` should already be at `/content/`. If not, upload it manually.

In [ ]:
import time, tarfile

OLD_DIR = Path("/content/train_subset_75")

if CHAT_JSON.exists() or CHAT_FULL.exists():
    n_files = sum(1 for _ in DATA_DIR.rglob("*") if _.is_file())
    print(f"[skip] {DATA_DIR} already populated ({n_files:,} files)")
else:
    assert TAR_PATH.exists() or OLD_DIR.exists(), (
        f"Need either {TAR_PATH} or {OLD_DIR}.\n"
        "Upload `train_subset_75.tar` to /content/ (e.g. via Colab file panel) then re-run."
    )
    DATA_DIR.parent.mkdir(parents=True, exist_ok=True)

    if OLD_DIR.exists() and any(OLD_DIR.iterdir()):
        print(f"[move] {OLD_DIR} -> {DATA_DIR}")
        if DATA_DIR.exists():
            DATA_DIR.rmdir()
        OLD_DIR.rename(DATA_DIR)
    else:
        with tarfile.open(TAR_PATH) as tf:
            tops = set()
            for i, m in enumerate(tf):
                tops.add(m.name.split("/", 1)[0])
                if i > 20:
                    break
        has_wrapper = len(tops) == 1 and next(iter(tops)) not in {"", "."}
        strip_flag = "--strip-components 1" if has_wrapper else ""
        print(f"[tar] top-level sample : {sorted(tops)[:5]}{'...' if len(tops) > 5 else ''}")
        print(f"[tar] strip wrapper dir? {has_wrapper}")
        DATA_DIR.mkdir(parents=True, exist_ok=True)
        print(f"[tar] extracting -> {DATA_DIR}  ({TAR_PATH.stat().st_size / 1e9:.2f} GB)")
        t0 = time.time()
        !tar -xf "{TAR_PATH}" {strip_flag} -C "{DATA_DIR}"
        print(f"[tar] done in {time.time()-t0:.1f}s")

n_imgs = sum(1 for p in DATA_DIR.rglob("*.jpg")) + sum(1 for p in DATA_DIR.rglob("*.png"))
print(f"[laion] images on disk: {n_imgs:,}")

## 4b. Upload + untar `coco_500.tar.gz`

Upload `coco_500.tar.gz` to `/content/` (Colab file panel, or run the cell below to use the interactive uploader). The archive is expected to contain `images/<COCO_val2014_*.jpg>` and `index_500.json` (the schema is `{image_id, image, text, label[5]}` per entry — already what we use elsewhere).

Idempotent: skips if `index_500.json` already exists in `/content/data/coco_500/`.

In [ ]:
# Optional interactive uploader. Comment out if you uploaded via the file panel.
if not COCO_TAR.exists() and not (COCO_DIR / "index_500.json").exists():
    try:
        from google.colab import files
        print("Upload coco_500.tar.gz now (or cancel and upload via the file panel):")
        up = files.upload()
        for name in up:
            src = Path("/content") / name
            if src.name != COCO_TAR.name:
                src.rename(COCO_TAR)
            print(f"[upload] -> {COCO_TAR}  ({COCO_TAR.stat().st_size / 1e6:.1f} MB)")
    except Exception as e:
        print(f"[upload] interactive upload unavailable: {e}")

In [ ]:
INDEX_JSON = COCO_DIR / "index_500.json"

if INDEX_JSON.exists():
    n_imgs = sum(1 for _ in (COCO_DIR / "images").glob("*")) if (COCO_DIR / "images").exists() else 0
    print(f"[skip] {COCO_DIR} already extracted ({n_imgs:,} images, index_500.json present)")
else:
    assert COCO_TAR.exists(), (
        f"Need {COCO_TAR}. Upload `coco_500.tar.gz` to /content/ and re-run."
    )
    # Peek to decide whether to strip a wrapper dir.
    with tarfile.open(COCO_TAR, "r:*") as tf:
        tops = set()
        for i, m in enumerate(tf):
            tops.add(m.name.split("/", 1)[0])
            if i > 20:
                break
    has_wrapper = len(tops) == 1 and next(iter(tops)) not in {"", "."}
    strip_flag = "--strip-components 1" if has_wrapper else ""
    print(f"[coco-tar] top-level sample : {sorted(tops)[:5]}{'...' if len(tops) > 5 else ''}")
    print(f"[coco-tar] strip wrapper dir? {has_wrapper}")
    COCO_DIR.mkdir(parents=True, exist_ok=True)
    print(f"[coco-tar] extracting -> {COCO_DIR}  ({COCO_TAR.stat().st_size / 1e6:.1f} MB)")
    t0 = time.time()
    !tar -xzf "{COCO_TAR}" {strip_flag} -C "{COCO_DIR}"
    print(f"[coco-tar] done in {time.time()-t0:.1f}s")

assert INDEX_JSON.exists(), f"Extraction finished but {INDEX_JSON} not found. Check archive layout."
n_imgs = sum(1 for _ in (COCO_DIR / "images").glob("*"))
print(f"[coco] images: {n_imgs:,}    index_500.json: OK")

## 4c. Split LAION held-out (seeded, idempotent)

Same split convention as `ABLATIONS.ipynb`: pulls `N_HELDOUT` examples out of `chat.json`, preserves the original as `chat_full.json` (sentinel), and writes the held-out set to `chat_heldout.json`. If you already split during training (e.g. via ABLATIONS), this cell is a no-op.

In [ ]:
import json, random

if CHAT_FULL.exists() and HELDOUT_JSON.exists():
    with open(CHAT_JSON) as f:    n_train = len(json.load(f))
    with open(HELDOUT_JSON) as f:  n_held  = len(json.load(f))
    print(f"[skip] split already done — train={n_train:,}  heldout={n_held:,}")
else:
    assert CHAT_JSON.exists(), f"Expected {CHAT_JSON} — run the untar cell first."
    with open(CHAT_JSON) as f:
        entries = json.load(f)
    assert len(entries) > N_HELDOUT, f"chat.json only has {len(entries):,} rows; need > {N_HELDOUT}"

    rng = random.Random(SPLIT_SEED)
    idx = list(range(len(entries)))
    rng.shuffle(idx)
    held_idx = set(idx[:N_HELDOUT])
    held  = [entries[i] for i in idx[:N_HELDOUT]]
    train = [entries[i] for i in range(len(entries)) if i not in held_idx]

    CHAT_JSON.rename(CHAT_FULL)
    with open(CHAT_JSON, "w") as f:    json.dump(train, f)
    with open(HELDOUT_JSON, "w") as f: json.dump(held,  f)

    print(f"[split] seed={SPLIT_SEED}")
    print(f"[split] full     : {CHAT_FULL}    ({len(entries):,} rows, preserved)")
    print(f"[split] train    : {CHAT_JSON}    ({len(train):,} rows)")
    print(f"[split] heldout  : {HELDOUT_JSON}  ({len(held):,} rows)")

## 5. Imports + dataset adapters

Every example is normalized to `{id, image_path, prompt, gold_refs}` so downstream code is dataset-agnostic. Swap in a new test set by writing one more `load_*()` returning that shape.

In [ ]:
import json, math, copy, time, random
from typing import List, Dict, Tuple, Optional

import numpy as np
import torch
import torch.nn.functional as F
from PIL import Image
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

from prismatic import load

DEVICE = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
DTYPE  = torch.bfloat16

torch.manual_seed(SPLIT_SEED)
random.seed(SPLIT_SEED)
np.random.seed(SPLIT_SEED)


def _flatten_conv(conv) -> Tuple[str, str]:
    """LLaVA-style conversation -> (human, gpt). Strips `<image>` placeholder."""
    if isinstance(conv, str):
        conv = json.loads(conv)
    human = next((t["value"] for t in conv if t.get("from") == "human"), "")
    gpt   = next((t["value"] for t in conv if t.get("from") == "gpt"), "")
    return human.replace("<image>", "").strip(), gpt.strip()


def load_laion_heldout(n: int = N_HELDOUT) -> List[dict]:
    """Held-out LAION examples (from chat_heldout.json — split done in cell 4c)."""
    assert HELDOUT_JSON.exists(), f"{HELDOUT_JSON} missing — run the split cell."
    with open(HELDOUT_JSON) as f:
        raw = json.load(f)
    out = []
    for ex in raw:
        img_rel = ex.get("image")
        if not img_rel:
            continue
        img_path = DATA_DIR / img_rel
        if not img_path.is_file():
            continue
        human, gold = _flatten_conv(ex.get("conversations", []))
        if not human or not gold:
            continue
        out.append({
            "id": str(ex.get("id") or img_rel),
            "image_path": img_path,
            "prompt": human,
            "gold_refs": [gold],
        })
        if len(out) >= n:
            break
    return out


def load_coco_500(n: int = N_COCO) -> List[dict]:
    with open(INDEX_JSON) as f:
        raw = json.load(f)
    out = []
    for ex in raw[:n]:
        img_path = COCO_DIR / "images" / ex["image"]
        if not img_path.is_file():
            continue
        out.append({
            "id": str(ex.get("image_id") or ex.get("image")),
            "image_path": img_path,
            "prompt": ex.get("text", "Describe this image in detail."),
            "gold_refs": [s.strip() for s in ex["label"]],
        })
    return out


laion_set = load_laion_heldout()
coco_set  = load_coco_500()
print(f"LAION held-out: {len(laion_set)}    COCO-500: {len(coco_set)}")
print("Sample LAION:", {k: (str(v) if hasattr(v, '__fspath__') else v) for k, v in laion_set[0].items()})
print("Sample COCO :", {k: (str(v) if hasattr(v, '__fspath__') else v) for k, v in coco_set[0].items()})

## 6. Model load + inference helpers

`load_vlm` wraps `prismatic.load`. `_generate` is the same pattern as `scripts/eval/run_basic_eval.py` — calls `super(type(vlm), vlm).generate(...)` to bypass `PrismaticVLM.generate`'s batch-1 wrapper.

In [ ]:
def _patch_vision_backbone_unwrap(vlm):
    """Newer timm's `get_intermediate_layers` returns a `list`, but `unpack_tuple` in
    base_vision.py only collapses `tuple`s, so `vlm.vision_backbone(pv)` leaks a 1-element
    list to the projector and trips the first `nn.Linear` with
    'argument input must be Tensor, not list'. Wrap the backbone's forward to collapse
    single-element list/tuple results to the inner tensor."""
    vb = vlm.vision_backbone
    if getattr(vb, "_unwrap_patched", False):
        return
    _orig_forward = vb.forward

    def forward(pixel_values, *args, **kwargs):
        out = _orig_forward(pixel_values, *args, **kwargs)
        if isinstance(out, (list, tuple)) and len(out) == 1:
            return out[0]
        return out

    vb.forward = forward
    vb._unwrap_patched = True


def load_vlm(ckpt_dir: Path):
    vlm = load(str(ckpt_dir), hf_token=os.environ.get("HF_TOKEN"))
    vlm.to(DEVICE, dtype=DTYPE)
    vlm.eval()
    _patch_vision_backbone_unwrap(vlm)
    return vlm


def is_selector_model(vlm) -> bool:
    from prismatic.util.selector import SelectorCompressorPipeline
    return isinstance(vlm.projector, SelectorCompressorPipeline)


def _pixel_values(vlm, image: Image.Image):
    pv = vlm.vision_backbone.image_transform(image)
    model_dtype = next(vlm.vision_backbone.parameters()).dtype
    if isinstance(pv, torch.Tensor):
        return pv[None, ...].to(device=DEVICE, dtype=model_dtype)
    if isinstance(pv, dict):
        return {k: v[None, ...].to(device=DEVICE, dtype=model_dtype) for k, v in pv.items()}
    raise ValueError(f"Unsupported pixel_values type: {type(pv)}")


@torch.inference_mode()
def _generate(vlm, pixel_values, human: str, max_new_tokens: int = MAX_NEW_TOKENS) -> str:
    tokenizer = vlm.llm_backbone.tokenizer
    pb = vlm.get_prompt_builder()
    pb.add_turn("human", human)
    prompt_text = pb.get_prompt()
    input_ids = tokenizer(prompt_text, truncation=True, return_tensors="pt").input_ids.to(vlm.device)
    autocast_dtype = vlm.llm_backbone.half_precision_dtype
    use_amp = (DEVICE.type == "cuda" and vlm.enable_mixed_precision_training)
    with torch.autocast("cuda", dtype=autocast_dtype, enabled=use_amp):
        gen_ids = super(type(vlm), vlm).generate(
            input_ids=input_ids,
            pixel_values=pixel_values,
            do_sample=False,
            max_new_tokens=max_new_tokens,
            min_length=1,
            no_repeat_ngram_size=3,
            repetition_penalty=1.15,
            early_stopping=True,
        )
    return tokenizer.decode(gen_ids[0, input_ids.shape[1]:], skip_special_tokens=True).strip()


def caption_dataset(vlm, examples: List[dict], desc: str = "gen") -> List[dict]:
    records = []
    for ex in tqdm(examples, desc=desc):
        try:
            image = Image.open(ex["image_path"]).convert("RGB")
            pv = _pixel_values(vlm, image)
            pred = _generate(vlm, pv, ex["prompt"])
        except Exception as e:
            records.append({
                "id": ex["id"],
                "image_path": str(ex["image_path"]),
                "prompt": ex["prompt"],
                "gold_refs": ex["gold_refs"],
                "pred": "",
                "error": f"{type(e).__name__}: {e}",
            })
            continue
        records.append({
            "id": ex["id"],
            "image_path": str(ex["image_path"]),
            "prompt": ex["prompt"],
            "gold_refs": ex["gold_refs"],
            "pred": pred,
        })
    return records

## 7. Selector vs. oracle top-K accuracy

Oracle = average LLM self-attention from response tokens → visual tokens, over early layers (matches `scripts/eval/precompute_oracle.py`). For each held-out LAION example we cache the selector's top-`K_MAX` and the oracle's top-`K_MAX`, then compute `acc@K = |sel∩oracle| / K` for each K in `K_VALUES`.

In [ ]:
@torch.inference_mode()
def selector_scores(vlm, pixel_values) -> torch.Tensor:
    """Per-patch keep-probabilities (N,) — mirrors the inference routing in
    SelectorCompressorPipeline._forward_inference (route_mode='selector'):
    runs MHA + residual before the router so the router operates on the same
    attention-enriched representation it was trained on."""
    patch_feats = vlm.vision_backbone(pixel_values)
    x = vlm.projector.vit2llm(patch_feats)
    attn_out, _ = vlm.projector.selector.mha(query=x, key=x, value=x, need_weights=False)
    context_x = x + attn_out
    logits = vlm.projector.selector.router(context_x)
    return F.softmax(logits, dim=-1)[:, :, 0].squeeze(0).float().cpu()


@torch.inference_mode()
def oracle_scores(vlm, image: Image.Image, human: str, gpt: str,
                  early_layers=(0, 1, 2, 3)) -> torch.Tensor:
    """Per-patch oracle scores (V,) — averaged LLM attention from response→visual
    over early layers. Continuous version of `oracle_indices`."""
    tokenizer = vlm.llm_backbone.tokenizer
    pb = vlm.get_prompt_builder()
    pb.add_turn(role="human", message=human)
    pb.add_turn(role="gpt",   message=gpt)
    full_text = pb.get_prompt()

    input_ids = tokenizer(full_text, truncation=True, return_tensors="pt").input_ids.to(vlm.device)
    attention_mask = torch.ones_like(input_ids, dtype=torch.bool)
    labels = input_ids.clone()
    labels[:, 0] = -100
    pv = _pixel_values(vlm, image)

    out = vlm(
        input_ids=input_ids,
        attention_mask=attention_mask,
        pixel_values=pv,
        labels=labels,
        output_attentions=True,
        return_dict=True,
    )
    attns = out.attentions
    if attns is None:
        raise RuntimeError("LLM returned no attentions; ensure eager attention is enabled.")

    patch_feats = vlm.vision_backbone(pv)
    projected = vlm.projector(patch_feats)
    V = projected.shape[1]

    seq = attns[0].shape[-1]
    visual_slice  = slice(1, 1 + V)
    response_slice = slice(1 + V, seq)
    if response_slice.stop - response_slice.start == 0:
        response_slice = slice(seq - 1, seq)

    layer_attns = [attns[i] for i in early_layers if i < len(attns)]
    stacked = torch.stack(layer_attns, dim=0)
    avg = stacked.float().mean(dim=(0, 2))
    attn_to_visual = avg[0, response_slice, visual_slice]
    return attn_to_visual.mean(dim=0).float().cpu()       # (V,)


def oracle_indices(vlm, image: Image.Image, human: str, gpt: str,
                   top_k: int, early_layers=(0, 1, 2, 3)) -> torch.LongTensor:
    """Top-k visual indices by LLM attention (response→visual)."""
    scores = oracle_scores(vlm, image, human, gpt, early_layers=early_layers)
    actual_k = min(top_k, scores.numel())
    _, top_idx = torch.topk(scores, k=actual_k, largest=True)
    return top_idx.to(torch.long)


def topk_intersection(a: torch.LongTensor, b: torch.LongTensor) -> int:
    return len(set(a.tolist()) & set(b.tolist()))

In [ ]:
# Load selector model; force eager attention so oracle extraction sees attentions.
vlm_sel = load_vlm(CKPT_DIR_SELECTOR)
assert is_selector_model(vlm_sel), "CKPT_DIR_SELECTOR doesn't look like a selector checkpoint."

inner_llm = vlm_sel.llm_backbone.llm
inner_llm.config._attn_implementation = "eager"
if hasattr(inner_llm.config, "attn_implementation"):
    inner_llm.config.attn_implementation = "eager"

# Force `full` route for oracle (LLM must see every projected token).
vlm_sel.projector.route_mode = "full"
vlm_sel.projector.use_qformer = False

K_MAX = max(K_VALUES)
agreement: Dict[int, List[float]] = {K: [] for K in K_VALUES}
selector_top_cache: Dict[str, torch.Tensor] = {}
oracle_top_cache:    Dict[str, torch.Tensor] = {}
oracle_scores_cache: Dict[str, torch.Tensor] = {}     # continuous (V,) per id
scores_cache:        Dict[str, torch.Tensor] = {}     # selector keep-probs (N,) per id


def _topk_indices(scores: torch.Tensor, k: int) -> torch.LongTensor:
    """Indices of the k highest-scoring entries (order doesn't matter to the caller)."""
    actual_k = min(k, scores.numel())
    return torch.topk(scores, k=actual_k, largest=True).indices.to(torch.long)


def _accuracy_at_K(sel_scores: torch.Tensor, ora_scores: torch.Tensor, K: int) -> float:
    """|argmax_K(selector) ∩ argmax_K(oracle)| / K  — fraction of selector's top-K
    that are also in the oracle's top-K (equivalently, the symmetric set IoU
    when both sets have the same size)."""
    sel_topK = set(_topk_indices(sel_scores, K).tolist())
    ora_topK = set(_topk_indices(ora_scores, K).tolist())
    return len(sel_topK & ora_topK) / K


errors = 0
for ex in tqdm(laion_set, desc="sel-vs-oracle"):
    try:
        image = Image.open(ex["image_path"]).convert("RGB")
        pv = _pixel_values(vlm_sel, image)

        scores     = selector_scores(vlm_sel, pv)                                          # (N,)
        ora_scores = oracle_scores(vlm_sel, image, ex["prompt"], ex["gold_refs"][0])       # (V,)
        # Sanity check: selector and oracle should be over the same V patch tokens.
        assert scores.numel() == ora_scores.numel(), (
            f"selector N={scores.numel()} != oracle V={ora_scores.numel()} — "
            "set route_mode='full', use_qformer=False on the selector projector."
        )

        scores_cache[ex["id"]]        = scores
        oracle_scores_cache[ex["id"]] = ora_scores
        # Cache top-K_MAX for downstream visualization (e.g. mask overlays).
        selector_top_cache[ex["id"]]  = _topk_indices(scores,     K_MAX)
        oracle_top_cache[ex["id"]]    = _topk_indices(ora_scores, K_MAX)

        # acc@K = |top-K(selector) ∩ top-K(oracle)| / K  — recomputed per K so the
        # comparison is unambiguous (no slicing-of-sorted-topk dependence).
        for K in K_VALUES:
            agreement[K].append(_accuracy_at_K(scores, ora_scores, K))
    except Exception as e:
        errors += 1
        if errors <= 3:
            print(f"[skip] {ex['id']}: {type(e).__name__}: {e}")

print(f"\nComputed agreement on {len(agreement[K_VALUES[0]])} examples ({errors} errors).")
acc_table = {
    K: {"mean": float(np.mean(v)) if v else float('nan'),
        "std":  float(np.std(v))  if v else float('nan')}
    for K, v in agreement.items()
}
print("\nSelector vs Oracle top-K accuracy:")
print(f"  (random baseline ≈ K/V where V = {next(iter(scores_cache.values())).numel()})")
for K, s in acc_table.items():
    V = next(iter(scores_cache.values())).numel()
    print(f"  K={K:>4d}   mean={s['mean']:.3f}   std={s['std']:.3f}   (random≈{K/V:.3f})")

In [ ]:
# Per-example agreement histograms
fig, axes = plt.subplots(1, len(K_VALUES), figsize=(4 * len(K_VALUES), 3), sharey=True)
if len(K_VALUES) == 1:
    axes = [axes]
for ax, K in zip(axes, K_VALUES):
    ax.hist(agreement[K], bins=20, range=(0, 1), edgecolor="black")
    ax.set_title(f"K={K}  mean={np.mean(agreement[K]):.3f}")
    ax.set_xlabel("accuracy")
axes[0].set_ylabel("# examples")
plt.tight_layout(); plt.show()

## 8. Selector heatmap visualization

Per-patch keep probabilities reshaped to (H,W) and bilinearly upsampled to image size. Adjacent panels: continuous heatmap, then a binary top-K mask for each K in `K_VALUES`.

In [ ]:
def _grid_hw(N: int) -> Tuple[int, int]:
    s = int(round(math.sqrt(N)))
    if s * s == N:
        return s, s
    raise ValueError(f"Cannot reshape {N} patch tokens into a square grid.")


def _heatmap_overlay(ax, image: Image.Image, scores_2d: np.ndarray, alpha: float = 0.5):
    W, H = image.size
    hm = torch.tensor(scores_2d)[None, None, ...].float()
    hm = F.interpolate(hm, size=(H, W), mode="bilinear", align_corners=False)[0, 0].numpy()
    ax.imshow(image)
    ax.imshow(hm, cmap="jet", alpha=alpha)
    ax.axis("off")


def _topk_mask(scores: torch.Tensor, K: int) -> torch.Tensor:
    m = torch.zeros_like(scores)
    m[torch.argsort(scores, descending=True)[:K]] = 1.0
    return m


# Deterministic per-example random "scores" so the top-K masks displayed for
# the random baseline are nested across K's (and reproducible).
def _random_scores_for(ex) -> torch.Tensor:
    n = scores_cache[ex["id"]].numel()
    g = torch.Generator().manual_seed(abs(hash(ex["id"])) % (2**31))
    return torch.rand(n, generator=g)


viz_examples = laion_set[:VIZ_N_SAMPLES]
ncols = 2 + len(K_VALUES)

def _render(title: str, score_fn, score_label: str):
    fig, axes = plt.subplots(len(viz_examples), ncols,
                             figsize=(2.6 * ncols, 2.6 * len(viz_examples)))
    if len(viz_examples) == 1:
        axes = [axes]
    fig.suptitle(title, fontsize=13, y=1.0)
    for row, ex in zip(axes, viz_examples):
        image = Image.open(ex["image_path"]).convert("RGB")
        scores = score_fn(ex)
        N = scores.numel(); H, W = _grid_hw(N)
        row[0].imshow(image); row[0].axis("off"); row[0].set_title("image", fontsize=9)
        _heatmap_overlay(row[1], image, scores.reshape(H, W).numpy())
        row[1].set_title(score_label, fontsize=9)
        for col, K in enumerate(K_VALUES, start=2):
            mask = _topk_mask(scores, K)
            _heatmap_overlay(row[col], image, mask.reshape(H, W).numpy(), alpha=0.55)
            row[col].set_title(f"top-{K}", fontsize=9)
    plt.tight_layout(); plt.show()


_render("Trained selector",        lambda ex: scores_cache[ex["id"]],        "keep prob")
_render("Oracle (LLM-attention)",  lambda ex: oracle_scores_cache[ex["id"]], "oracle score")
_render("Random baseline",         _random_scores_for,                       "random ~ U(0,1)")

## 9. End-to-end captioning — PHI2-Selector × K sweep  + Random baseline × K sweep

Runs both routing modes on the same loaded model (no extra checkpoint load). The random baseline uses `route_mode="random_topk"` — same Q-Former pipeline, just non-learned token selection — so the difference vs. the trained selector isolates the value of learned routing.

In [ ]:
import json

def _save_records_atomic(records, path):
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    with open(tmp, "w") as f:
        json.dump(records, f, indent=2, default=str)
    tmp.replace(path)


# Reset Q-Former (cell 20 turned it off for the oracle pass).
vlm_sel.projector.use_qformer = True

records_sel:  Dict[str, Dict[int, List[dict]]] = {"laion": {}, "coco": {}}
records_rand: Dict[str, Dict[int, List[dict]]] = {"laion": {}, "coco": {}}

modes = [
    ("selector",    "sel",  records_sel),
    ("random_topk", "rand", records_rand),
]
for route_mode, slug, store in modes:
    vlm_sel.projector.route_mode = route_mode
    for K in K_VALUES:
        vlm_sel.projector.inference_k = K
        for ds_name, ds in [("laion", laion_set), ("coco", coco_set)]:
            out_path = EVAL_OUT_DIR / f"{slug}_K={K}_{ds_name}.json"
            if out_path.exists():
                with open(out_path) as f:
                    store[ds_name][K] = json.load(f)
                print(f"[skip ] {out_path.name}  ({len(store[ds_name][K])} records)")
                continue
            recs = caption_dataset(vlm_sel, ds, desc=f"{slug}-K={K} {ds_name}")
            _save_records_atomic(recs, out_path)
            store[ds_name][K] = recs
            print(f"[saved] {out_path.name}  ({len(recs)} records)")

del vlm_sel
if DEVICE.type == "cuda":
    torch.cuda.empty_cache()

## 10. End-to-end captioning — PHI2-MLP

In [ ]:
import json

def _save_records_atomic(records, path):
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    with open(tmp, "w") as f:
        json.dump(records, f, indent=2, default=str)
    tmp.replace(path)


records_mlp: Dict[str, List[dict]] = {}
vlm_mlp = None
for ds_name, ds in [("laion", laion_set), ("coco", coco_set)]:
    out_path = EVAL_OUT_DIR / f"mlp_{ds_name}.json"
    if out_path.exists():
        with open(out_path) as f:
            records_mlp[ds_name] = json.load(f)
        print(f"[skip ] {out_path.name}  ({len(records_mlp[ds_name])} records)")
        continue
    if vlm_mlp is None:
        vlm_mlp = load_vlm(CKPT_DIR_MLP)
    recs = caption_dataset(vlm_mlp, ds, desc=f"mlp {ds_name}")
    _save_records_atomic(recs, out_path)
    records_mlp[ds_name] = recs
    print(f"[saved] {out_path.name}  ({len(recs)} records)")

if vlm_mlp is not None:
    del vlm_mlp
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

## 11. Metrics — CIDEr + CLIPScore

- **CIDEr** via `pycocoevalcap.cider.cider.Cider`.
- **CLIPScore** = `2.5 * max(0, cos(img_emb, text_emb))` averaged across examples (Hessel et al., 2021), using `open_clip` ViT-B/32 OpenAI weights.

In [ ]:
from pycocoevalcap.cider.cider import Cider
import open_clip

_clip_model, _, _clip_preprocess = open_clip.create_model_and_transforms("ViT-B-32", pretrained="openai")
_clip_tokenizer = open_clip.get_tokenizer("ViT-B-32")
_clip_model = _clip_model.to(DEVICE).eval()


def compute_cider(records: List[dict]) -> float:
    gts = {r["id"]: list(r["gold_refs"]) for r in records if r.get("pred") and r.get("gold_refs")}
    res = {r["id"]: [r["pred"]]         for r in records if r.get("pred") and r.get("gold_refs")}
    if not gts:
        return float("nan")
    score, _ = Cider().compute_score(gts, res)
    return float(score)


@torch.inference_mode()
def compute_clipscore(records: List[dict], batch_size: int = 32) -> float:
    valid = [r for r in records if r.get("pred")]
    if not valid:
        return float("nan")
    sims = []
    for i in range(0, len(valid), batch_size):
        chunk = valid[i:i + batch_size]
        imgs = torch.stack([_clip_preprocess(Image.open(r["image_path"]).convert("RGB")) for r in chunk]).to(DEVICE)
        toks = _clip_tokenizer([r["pred"] for r in chunk]).to(DEVICE)
        img_emb = _clip_model.encode_image(imgs); img_emb = img_emb / img_emb.norm(dim=-1, keepdim=True)
        txt_emb = _clip_model.encode_text(toks);  txt_emb = txt_emb / txt_emb.norm(dim=-1, keepdim=True)
        sims.append((img_emb * txt_emb).sum(dim=-1).clamp(min=0).cpu())
    sims = torch.cat(sims)
    return float(2.5 * sims.mean().item())


def score(records: List[dict]) -> Dict[str, float]:
    return {"cider": compute_cider(records), "clipscore": compute_clipscore(records)}


metrics: Dict[str, Dict[str, float]] = {}
metrics["PHI2-MLP / laion"] = score(records_mlp["laion"])
metrics["PHI2-MLP / coco"]  = score(records_mlp["coco"])
for K in K_VALUES:
    metrics[f"PHI2-Selector@K={K} / laion"] = score(records_sel["laion"][K])
    metrics[f"PHI2-Selector@K={K} / coco"]  = score(records_sel["coco"][K])
for K in K_VALUES:
    metrics[f"PHI2-Random@K={K} / laion"]   = score(records_rand["laion"][K])
    metrics[f"PHI2-Random@K={K} / coco"]    = score(records_rand["coco"][K])

print(f"{'variant':<40s}  {'CIDEr':>8s}  {'CLIPScore':>10s}")
print("-" * 64)
for name, m in metrics.items():
    print(f"{name:<40s}  {m['cider']:>8.3f}  {m['clipscore']:>10.3f}")

## 12. Qualitative examples — (image, prompt, gold, MLP-pred, Selector-pred)

Selector predictions shown at K = `DEMO_K_FOR_GEN`.

In [ ]:
def show_examples(ds_name: str, n: int = 6):
    mlp_recs = records_mlp[ds_name]
    sel_recs = records_sel[ds_name][DEMO_K_FOR_GEN]
    sel_by_id = {r["id"]: r for r in sel_recs}
    pairs = [(r, sel_by_id[r["id"]]) for r in mlp_recs if r["id"] in sel_by_id]
    rng = random.Random(SPLIT_SEED)
    picks = rng.sample(pairs, k=min(n, len(pairs)))
    fig, axes = plt.subplots(len(picks), 1, figsize=(8, 4 * len(picks)))
    if len(picks) == 1:
        axes = [axes]
    for ax, (rm, rs) in zip(axes, picks):
        ax.imshow(Image.open(rm["image_path"]).convert("RGB")); ax.axis("off")
        gold = rm["gold_refs"][0] if rm["gold_refs"] else ""
        cap = (f"[{ds_name}  id={rm['id']}]\n"
               f"prompt: {rm['prompt']}\n"
               f"gold:     {gold}\n"
               f"MLP:      {rm['pred']}\n"
               f"Sel@{DEMO_K_FOR_GEN}: {rs['pred']}")
        ax.set_title(cap, fontsize=9, loc="left", wrap=True)
    plt.tight_layout(); plt.show()

show_examples("laion", n=6)
show_examples("coco",  n=6)

## 13. Save artifacts to Drive

Writes the metrics table, K-sweep accuracy, and all (id, prompt, gold, pred) records into a single JSON under `runs/<EVAL_TAG>/eval_results.json` on Drive — durable across session teardowns.

In [ ]:
payload = {
    "config": {
        "run_id_mlp":       RUN_ID_MLP,
        "run_id_selector":  RUN_ID_SELECTOR,
        "ckpt_dir_mlp":     str(CKPT_DIR_MLP),
        "ckpt_dir_selector":str(CKPT_DIR_SELECTOR),
        "n_laion":          len(laion_set),
        "n_coco":           len(coco_set),
        "K_values":         K_VALUES,
        "max_new_tokens":   MAX_NEW_TOKENS,
        "split_seed":       SPLIT_SEED,
    },
    "metrics": metrics,
    "selector_vs_oracle_accuracy": acc_table,
    "records": {
        "mlp":      records_mlp,
        "selector": {ds: {str(K): records_sel[ds][K]  for K in K_VALUES} for ds in ("laion", "coco")},
        "random":   {ds: {str(K): records_rand[ds][K] for K in K_VALUES} for ds in ("laion", "coco")},
    },
}

RESULTS_JSON.parent.mkdir(parents=True, exist_ok=True)
tmp = RESULTS_JSON.with_suffix(".json.tmp")
with open(tmp, "w") as f:
    json.dump(payload, f, indent=2, default=str)
tmp.replace(RESULTS_JSON)
print(f"wrote {RESULTS_JSON}  ({RESULTS_JSON.stat().st_size / 1024:.1f} KB)")

## 14. How to extend

- **New test set** — write `load_<name>()` returning `{id, image_path, prompt, gold_refs}` and feed it through `caption_dataset` + `score`. Nothing else changes.
- **Different K's** — edit `K_VALUES` in cell 2 and re-run from cell 7 (sel-vs-oracle) onward.
- **Third checkpoint** — add `RUN_ID_X` + `CKPT_DIR_X` in cell 2, replicate cell 10's pattern.
- **Subsampling for speed** — lower `N_HELDOUT` / `N_COCO`. Runtime ≈ `(1 MLP + len(K_VALUES) × Selector) × (N_HELDOUT + N_COCO)` generations + one forward per LAION example for the oracle.
- **Generation knobs** — tweak `MAX_NEW_TOKENS` and the `no_repeat_ngram_size` / `repetition_penalty` in `_generate` (cell 6).
- **Sanity check** — a freshly initialized selector has ~uniform keep-probs and `acc@K ≈ K/N` (random). A trained selector should be visibly better than that floor.